# Natural Language Processing

# Retrieval-Augmented generation (RAG)

RAG is a technique for augmenting LLM knowledge with additional, often private or real-time, data.

LLMs can reason about wide-ranging topics, but their knowledge is limited to the public data up to a specific point in time that they were trained on. If you want to build AI applications that can reason about private data or data introduced after a model’s cutoff date, you need to augment the knowledge of the model with the specific information it needs.



Introducing `IshikaBot`, an innovative chatbot designed to assist Chaky (the instructor) and TA (Gun) in explaining the lesson of the NLP course to students. Leveraging LangChain technology, ChakyBot excels in retrieving information from documents, ensuring a seamless and efficient learning experience for students engaging with the NLP curriculum.

1. Prompt
2. Retrieval
3. Memory
4. Chain

In [12]:
import os
import torch
# Set GPU device
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# os.environ['http_proxy']  = 'http://192.41.170.23:3128'
# os.environ['https_proxy'] = 'http://192.41.170.23:3128'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

## 1. Prompt

A set of instructions or input provided by a user to guide the model's response, helping it understand the context and generate relevant and coherent language-based output, such as answering questions, completing sentences, or engaging in a conversation.

In [13]:
from langchain import PromptTemplate

prompt_template = """
   I'm a friendly chatbot here to answer questions about Ishika based on provided documents.
I'll give gentle and informative responses about Ishika's life, education, work, and beliefs.
If I use a document to answer, I’ll cite it (e.g., resume, bio).
    {context}
    Question: {question}
    Answer:
    """.strip()


PROMPT = PromptTemplate.from_template(
    template = prompt_template
)

PROMPT
#using str.format 
#The placeholder is defined using curly brackets: {} {}

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="I'm a friendly chatbot here to answer questions about Ishika based on provided documents.\nI'll give gentle and informative responses about Ishika's life, education, work, and beliefs.\nIf I use a document to answer, I’ll cite it (e.g., resume, bio).\n    {context}\n    Question: {question}\n    Answer:")

In [14]:
PROMPT.format(
    context = "Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can effectively generalize and thus perform tasks without explicit instructions.",
    question = "What is Machine Learning"
)

"I'm a friendly chatbot here to answer questions about Ishika based on provided documents.\nI'll give gentle and informative responses about Ishika's life, education, work, and beliefs.\nIf I use a document to answer, I’ll cite it (e.g., resume, bio).\n    Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can effectively generalize and thus perform tasks without explicit instructions.\n    Question: What is Machine Learning\n    Answer:"

## 2. Retrieval

1. `Document loaders` : Load documents from many different sources (HTML, PDF, code). 
2. `Document transformers` : One of the essential steps in document retrieval is breaking down a large document into smaller, relevant chunks to enhance the retrieval process.
3. `Text embedding models` : Embeddings capture the semantic meaning of the text, allowing you to quickly and efficiently find other pieces of text that are similar.
4. `Vector stores`: there has emerged a need for databases to support efficient storage and searching of these embeddings.
5. `Retrievers` : Once the data is in the database, you still need to retrieve it.

### 2.1 Document Loaders 
Use document loaders to load data from a source as Document's. A Document is a piece of text and associated metadata. For example, there are document loaders for loading a simple .txt file, for loading the text contents of any web page, or even for loading a transcript of a YouTube video.

[PDF Loader](https://python.langchain.com/docs/modules/data_connection/document_loaders/pdf)



In [15]:
import os
folder_path = './personal'
if os.path.exists(folder_path):
    print(f"Folder exists. Contents: {os.listdir(folder_path)}")
else:
    print("Folder does not exist!")

Folder exists. Contents: ['cv_bio.pdf']


In [16]:
from langchain.document_loaders import PyMuPDFLoader, TextLoader
from langchain.document_loaders import DirectoryLoader

In [17]:
from langchain.document_loaders import PyMuPDFLoader

nlp_docs = './personal/cv_bio.pdf'

loader = PyMuPDFLoader(nlp_docs)
documents = loader.load()

c:\Users\Ishika\anaconda3\envs\assignment\Lib\site-packages\langchain_community\document_loaders\parsers\pdf.py:300: UserWarning: Warning: Empty content on page 1 of document ./personal/cv_bio.pdf
  warnings.warn(


In [18]:
print(f"Loaded {len(documents)} documents")

Loaded 2 documents


In [19]:
documents[1]

Document(metadata={'source': './personal/cv_bio.pdf', 'file_path': './personal/cv_bio.pdf', 'page': 1, 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'iLovePDF', 'creationDate': '', 'modDate': 'D:20250316060033Z', 'trapped': ''}, page_content='')

### 2.2 Document Transformers

This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough

In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100
)

doc = text_splitter.split_documents(documents)

In [21]:
doc[1]

Document(metadata={'source': './personal/cv_bio.pdf', 'file_path': './personal/cv_bio.pdf', 'page': 0, 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'iLovePDF', 'creationDate': '', 'modDate': 'D:20250316060033Z', 'trapped': ''}, page_content='while ensuring inclusivity and ethical considerations. \nWork Experience and Industry Involvement \nI have 3 years of work experience. My focus has been on research and practical projects within the \ntech industry. My interest lies in applying AI-driven solutions to real-world problems, making \ntechnology more accessible, ethical, and inclusive. \nRole of Technology in Society \nI strongly believe that technology should empower individuals and be a tool for solving real-world \nproblems efficiently. Innovation should prioritize accessibility and inclusivity, ensuring that \nadvancements benefit diverse communities without ethical compromises. \nCultural Values and Tech

In [22]:
len(doc)

5

### 2.3 Text Embedding Models
Embeddings create a vector representation of a piece of text. This is useful because it means we can think about text in the vector space, and do things like semantic search where we look for pieces of text that are most similar in the vector space.

*Note* Instructor Model : [Huggingface](gingface.co/hkunlp/instructor-base) | [Paper](https://arxiv.org/abs/2212.09741)

In [23]:
import torch
from langchain.embeddings import HuggingFaceInstructEmbeddings

model_name = 'hkunlp/instructor-base'

embedding_model = HuggingFaceInstructEmbeddings(
    model_name = model_name,
    model_kwargs = {"device" : device}
)

c:\Users\Ishika\anaconda3\envs\assignment\Lib\site-packages\InstructorEmbedding\instructor.py:7: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import trange
c:\Users\Ishika\anaconda3\envs\assignment\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


load INSTRUCTOR_Transformer


c:\Users\Ishika\anaconda3\envs\assignment\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Ishika\anaconda3\envs\assignment\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


max_seq_length  512


### 2.4 Vector Stores

One of the most common ways to store and search over unstructured data is to embed it and store the resulting embedding vectors, and then at query time to embed the unstructured query and retrieve the embedding vectors that are 'most similar' to the embedded query. A vector store takes care of storing embedded data and performing vector search for you.

In [24]:
# 2.4 Vector Stores
import os
from langchain.vectorstores import FAISS

# Define vector store path
vector_path = '../vector-store'
if not os.path.exists(vector_path):
    os.makedirs(vector_path)
    print("Created vector-store directory")

# Create FAISS index from split documents
vectordb = FAISS.from_documents(
    documents=doc,  # From 2.2
    embedding=embedding_model  # From 2.3
)

# Save the FAISS index
vectordb.save_local(
    folder_path=os.path.join(vector_path, 'ishika_data'),
    index_name='personal'
)
print("FAISS index saved successfully")

FAISS index saved successfully


### 2.5 retrievers
A retriever is an interface that returns documents given an unstructured query. It is more general than a vector store. A retriever does not need to be able to store documents, only to return (or retrieve) them. Vector stores can be used as the backbone of a retriever, but there are other types of retrievers as well.

In [25]:
# 2.5 Retrievers
from langchain.vectorstores import FAISS

# Define vector store path (consistent with 2.4)
vector_path = '../vector-store'
db_file_name = 'ishika_data'

# Load the FAISS index
vectordb = FAISS.load_local(
    folder_path=os.path.join(vector_path, db_file_name),
    embeddings=embedding_model,  # From 2.3
    index_name='personal',      # Match 2.4
    allow_dangerous_deserialization=True  # Fix the ValueError
)

# Create retriever
retriever = vectordb.as_retriever()
print("Retriever loaded successfully")

Retriever loaded successfully


In [26]:
retriever.get_relevant_documents("What is Dependency Parsing")

C:\Users\Ishika\AppData\Local\Temp\ipykernel_16124\63918203.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retriever.get_relevant_documents("What is Dependency Parsing")


[Document(metadata={'source': './personal/cv_bio.pdf', 'file_path': './personal/cv_bio.pdf', 'page': 0, 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'iLovePDF', 'creationDate': '', 'modDate': 'D:20250316060033Z', 'trapped': ''}, page_content="Cultural Values and Technological Advancements \nCultural diversity should be respected in technological developments. I believe that technology \nshould not only be a driver of progress but also an enabler of cultural preservation, inclusivity, and \nethical use across different societies. \nChallenges in My Master’s Studies \nOne of the most challenging aspects of my master's studies is mastering complex topics such as \nmachine learning models, natural language processing (NLP), and deep learning. Understanding the \nmathematical foundations behind these models, optimizing their performance, and applying them to"),
 Document(metadata={'source': './personal/cv_bio.pdf

In [27]:
retriever.get_relevant_documents("What is Transformers")

[Document(metadata={'source': './personal/cv_bio.pdf', 'file_path': './personal/cv_bio.pdf', 'page': 0, 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'iLovePDF', 'creationDate': '', 'modDate': 'D:20250316060033Z', 'trapped': ''}, page_content='meaningful research that bridges the gap between AI and human-centric design.'),
 Document(metadata={'source': './personal/cv_bio.pdf', 'file_path': './personal/cv_bio.pdf', 'page': 0, 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'iLovePDF', 'creationDate': '', 'modDate': 'D:20250316060033Z', 'trapped': ''}, page_content='while ensuring inclusivity and ethical considerations. \nWork Experience and Industry Involvement \nI have 3 years of work experience. My focus has been on research and practical projects within the \ntech industry. My interest lies in applying AI-driven solutions to real-wo

## 3. Memory

One of the core utility classes underpinning most (if not all) memory modules is the ChatMessageHistory class. This is a super lightweight wrapper that provides convenience methods for saving HumanMessages, AIMessages, and then fetching them all.

You may want to use this class directly if you are managing memory outside of a chain.


In [28]:
from langchain.memory import ChatMessageHistory

history = ChatMessageHistory()
history

InMemoryChatMessageHistory(messages=[])

In [29]:
history.add_user_message('hi')
history.add_ai_message('Whats up?')
history.add_user_message('How are you')
history.add_ai_message('I\'m quite good. How about you?')

In [30]:
history

InMemoryChatMessageHistory(messages=[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content='Whats up?', additional_kwargs={}, response_metadata={}), HumanMessage(content='How are you', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm quite good. How about you?", additional_kwargs={}, response_metadata={})])

### 3.1 Memory types

There are many different types of memory. Each has their own parameters, their own return types, and is useful in different scenarios. 
- Converstaion Buffer
- Converstaion Buffer Window

What variables get returned from memory

Before going into the chain, various variables are read from memory. These have specific names which need to align with the variables the chain expects. You can see what these variables are by calling memory.load_memory_variables({}). Note that the empty dictionary that we pass in is just a placeholder for real variables. If the memory type you are using is dependent upon the input variables, you may need to pass some in.

In this case, you can see that load_memory_variables returns a single key, history. This means that your chain (and likely your prompt) should expect an input named history. You can usually control this variable through parameters on the memory class. For example, if you want the memory variables to be returned in the key chat_history you can do:

#### Converstaion Buffer
This memory allows for storing messages and then extracts the messages in a variable.

In [31]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()
memory.save_context({'input':'hi'}, {'output':'What\'s up?'})
memory.save_context({"input":'How are you?'},{'output': 'I\'m quite good. How about you?'})
memory.load_memory_variables({})

C:\Users\Ishika\AppData\Local\Temp\ipykernel_16124\1450517278.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


{'history': "Human: hi\nAI: What's up?\nHuman: How are you?\nAI: I'm quite good. How about you?"}

In [32]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(return_messages = True)
memory.save_context({'input':'hi'}, {'output':'What\'s up?'})
memory.save_context({"input":'How are you?'},{'output': 'I\'m quite good. How about you?'})
memory.load_memory_variables({})

{'history': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}),
  AIMessage(content="What's up?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='How are you?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="I'm quite good. How about you?", additional_kwargs={}, response_metadata={})]}

#### Conversation Buffer Window
- it keeps a list of the interactions of the conversation over time. 
- it only uses the last K interactions. 
- it can be useful for keeping a sliding window of the most recent interactions, so the buffer does not get too large.

from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=1)
memory.save_context({'input':'hi'}, {'output':'What\'s up?'})
memory.save_context({"input":'How are you?'},{'output': 'I\'m quite good. How about you?'})
memory.load_memory_variables({})

## 4. Chain

Using an LLM in isolation is fine for simple applications, but more complex applications require chaining LLMs - either with each other or with other components.

An `LLMChain` is a simple chain that adds some functionality around language models.
- it consists of a `PromptTemplate` and a `LM` (either an LLM or chat model).
- it formats the prompt template using the input key values provided (and also memory key values, if available), 
- it passes the formatted string to LLM and returns the LLM output.

Note : [Download Fastchat Model Here](https://huggingface.co/lmsys/fastchat-t5-3b-v1.0)

In [33]:
## groq API = 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

In [35]:
import os
model_path = './models/fastchat-t5-3b-v1.0/'
print("Full path:", os.path.abspath(model_path))
print("Exists:", os.path.exists(model_path))
if os.path.exists(model_path):
    print("Contents:", os.listdir(model_path))
else:
    print("Directory not found!")

Full path: d:\2nd SEM\NLP\NLP_Assignment\A6\models\fastchat-t5-3b-v1.0
Exists: True
Contents: ['.git', '.gitattributes', 'added_tokens.json', 'config.json', 'generation_config.json', 'model-00001-of-00003.safetensors', 'model-00002-of-00003.safetensors', 'model-00003-of-00003.safetensors', 'model.safetensors.index.json', 'pytorch_model.bin.index.json', 'README.md', 'special_tokens_map.json', 'spiece.model', 'tokenizer.json', 'tokenizer_config.json']


In [ ]:
# 4. Chain (Updated with Class ConversationalRetrievalChain)
from transformers import AutoTokenizer, pipeline, AutoModelForSeq2SeqLM
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain, ConversationalRetrievalChain
from langchain.chains.conversational_retrieval.prompts import CONDENSE_QUESTION_PROMPT
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains.question_answering import load_qa_chain
from langchain_groq import ChatGroq

# Load FastChat-T5 model (local)
model_id = './models/fastchat-t5-3b-v1.0/'
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token_id = tokenizer.eos_token_id
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device=0 if torch.cuda.is_available() else -1
)
llm = HuggingFacePipeline(pipeline=pipe)

# Load Groq's llama3-70b
llm_groq = ChatGroq(
    model_name="llama3-70b-8192",
    api_key="groq_API"  # Your Groq API key
)

# Load retriever
retriever = vectordb.as_retriever()

# Define components for FastChat-T5 chain
question_generator = LLMChain(llm=llm, prompt=CONDENSE_QUESTION_PROMPT, verbose=True)
doc_chain = load_qa_chain(llm=llm, chain_type='stuff', verbose=True)  # No PROMPT yet, using default
memory = ConversationBufferWindowMemory(k=3, memory_key="chat_history", return_messages=True, output_key='answer')

# FastChat-T5 chain
chain = ConversationalRetrievalChain(
    retriever=retriever,
    question_generator=question_generator,
    combine_docs_chain=doc_chain,
    return_source_documents=True,
    memory=memory,
    verbose=True,
    get_chat_history=lambda h: h
)

# Define components for Groq chain (reuse memory and retriever)
question_generator_groq = LLMChain(llm=llm_groq, prompt=CONDENSE_QUESTION_PROMPT, verbose=True)
doc_chain_groq = load_qa_chain(llm=llm_groq, chain_type='stuff', verbose=True)

# Groq Llama3 chain
chain_groq = ConversationalRetrievalChain(
    retriever=retriever,
    question_generator=question_generator_groq,
    combine_docs_chain=doc_chain_groq,
    return_source_documents=True,
    memory=memory,
    verbose=True,
    get_chat_history=lambda h: h
)

print("Original LLM: FastChat-T5-3B | Alternative LLM: Groq Llama3-70B")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 3/3 [01:16<00:00, 25.42s/it]
C:\Users\Ishika\AppData\Local\Temp\ipykernel_16124\1741277158.py:22: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


Original LLM: FastChat-T5-3B | Alternative LLM: Groq Llama3-70B


C:\Users\Ishika\AppData\Local\Temp\ipykernel_16124\1741277158.py:34: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  question_generator = LLMChain(llm=llm, prompt=CONDENSE_QUESTION_PROMPT, verbose=True)
C:\Users\Ishika\AppData\Local\Temp\ipykernel_16124\1741277158.py:35: LangChainDeprecationWarning: This class is deprecated. See the following migration guides for replacements based on `chain_type`:
stuff: https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain
map_reduce: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain
refine: https://python.langchain.com/docs/versions/migrating_chains/refine_chain
map_rerank: https://python.langchain.com/docs/versions/migrating_chains/map_rerank_docs_chain

See also guides on retrieval and question-answering here: https://python.langchain.com/docs/how_to/#qa-with-rag
  doc_cha

## 5. Chatbot

In [ ]:
prompt_question = "How old is Ishika?"
answer = chain({"question":prompt_question})
answer

C:\Users\Ishika\AppData\Local\Temp\ipykernel_11952\1267086776.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  answer = chain({"question":prompt_question})




> Entering new ConversationalRetrievalChain chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Name: Ishika Pradhan Age: 26  
Core Beliefs: 
I believe technology should empower individuals and solve real-world problems efficiently, fostering
innovation and accessibility. I am passionate about using tech to create solutions that enhance lives
while ensuring inclusivity and ethical considerations. 
Cultural Values: 
I strongly believe that technological advancements should respect cultural diversity, ensuring
inclusivity and ethical use across communities. Bridging the gap between technology and society is
essential to me. 
Education & Career Goals: 
I am currently pursuing a Master’s in Data science and AI at Asian Institute ofTtechnology . Balancing

coursewor

{'question': 'How old is Ishika?',
 'chat_history': [],
 'answer': '<pad>  26\n',
 'source_documents': [Document(metadata={'source': './personal/cv_bio.pdf', 'file_path': './personal/cv_bio.pdf', 'page': 1, 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'iLovePDF', 'creationDate': '', 'modDate': 'D:20250315082157Z', 'trapped': ''}, page_content='Name: Ishika Pradhan Age: 26  \nCore Beliefs: \nI believe technology should empower individuals and solve real-world problems efficiently, fostering\ninnovation and accessibility. I am passionate about using tech to create solutions that enhance lives\nwhile ensuring inclusivity and ethical considerations. \nCultural Values: \nI strongly believe that technological advancements should respect cultural diversity, ensuring\ninclusivity and ethical use across communities. Bridging the gap between technology and society is\nessential to me. \nEducation & Career Goals: \nI a

In [ ]:
# 5. Chatbot
questions = [
    "How old is Ishika?",
    "What is Ishika's highest education?"

]

print("Testing FastChat-T5-3B:")
for q in questions:
    result = chain({"question": q})
    print(f"Q: {q}")
    print(f"A: {result['answer']}\n")

print("Testing Groq Llama3-70B:")
for q in questions:
    result_groq = chain_groq({"question": q})
    print(f"Q: {q}")
    print(f"A: {result_groq['answer']}\n")

Testing FastChat-T5-3B:


> Entering new ConversationalRetrievalChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:
[HumanMessage(content='How old is Ishika?', additional_kwargs={}, response_metadata={}), AIMessage(content='<pad>  26\n', additional_kwargs={}, response_metadata={})]
Follow Up Input: How old is Ishika?
Standalone question:

> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Name: Ishika Pradhan Age: 26  
Core Beliefs: 
I believe technology should empower individuals and solve real-world problems efficiently, fostering
innovation and accessibility. I am passi

## Analysis and Problem Solving

###  List of the retriever and generator models used

1) Retriever Model
<br>
   - Model: FAISS (Fast Approximate Nearest Neighbors Index) with HuggingFaceInstructEmbeddings.
   <br>
Details:
   <br>
   - Embeddings: hkunlp/instructor-base.<br>
   - Vector Store: FAISS, initialized with FAISS.from_documents() and loaded with FAISS.load_local().

2) Generator Model
<br>
- Model: Groq Llama3-70B-8192.<br>


### Analyze any issues related to the models providing unrelated information. 

Building this chatbot was a journey, but I hit some bumps where the models spit out stuff that didn’t quite fit. Here’s what I noticed about the retriever and generator going off-track, plus some ideas to fix it.

- Embedding Model (HuggingFaceInstructEmbeddings)
I used hkunlp/instructor-base to make vectors out of my text, but it’s not perfect. Sometimes it doesn’t really “get” what I’m asking. Like, if I say, “What projects has Ishika done?” it might grab chunks about her school instead of work because the embeddings latch onto vague words like “experience.” It’s a general-purpose model, so it’s not tuned to Ishika’s life story—names like hers or quirky details might throw it off. Also, short questions like “What’s her deal?” can turn into fuzzy vectors that pull random bits.

    - Fixes: Maybe mix in a keyword search with FAISS—like BM25—to catch specific terms better. Or tweak the embeddings with some bio-specific training, though that’s a bit of a project.
- Vector Store (FAISS)
FAISS is awesome for speed, but omly with small files. If I ask something offbeat like “What’s Ishika’s favorite food?” it still picks the “closest” chunks—say, her education—since there’s no food info to find. It’s like forcing a square peg into a round hole. Plus, FAISS goes by vector math, so “Tell me about her job” might grab project details instead of her actual work history if the numbers line up wrong. And with only 4 chunks retrieved, it sometimes misses the good stuff—like details getting buried under noise.
    - Fixes: I could chop the PDF into smaller, overlapping pieces so each chunk has more context. Grabbing more than 4 chunks might help too, or adding a reranking step to sort out what’s actually useful.
- Generator model (Groq Llama3-70B)
The Groq model is a beast at writing, but it’s got a mind of its own. If FAISS hands it weak context—like education chunks for a hobbies question—it might guess “Ishika loves coding” when cv_bio.pdf says nothing about that. It’s also got this huge brain from general training, so it might sneak in random facts. Like, “What’s her job?” could get “She’s a software engineer” even if the bio says something else. I caught it once trying to add “(No document cited...)” before I yanked that out—it’s eager to sound smart, even when it shouldn’t. For “What’s her favorite color?” with no data, it might just pick “Blue” instead of shrugging.
    - Fixes: I’d tweak my prompt to say, “Stick to the docs or say ‘I don’t know.’” Maybe add a check—if the retrieved chunks don’t match well, it could just ask me to rephrase instead of making stuff up.
